# Lesson 7.6 - MCP and Agent2Agent Interoperability Demo

## Learning Objectives
- Simulate MCP-style tool discovery and invocation with approval checks.
- Simulate A2A-style async task collaboration between agents.
- Understand where MCP and A2A complement each other.

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Dict, Any, List
import uuid
import time

## 1) Minimal MCP Simulation

In [2]:
@dataclass
class MCPTool:
    name: str
    description: str
    func: Callable[[Dict[str, Any]], Dict[str, Any]]
    sensitive: bool = False

@dataclass
class MCPServer:
    name: str
    tools: Dict[str, MCPTool]

    def list_tools(self) -> List[Dict[str, Any]]:
        return [{'name': t.name, 'description': t.description, 'sensitive': t.sensitive} for t in self.tools.values()]

    def call_tool(self, tool_name: str, args: Dict[str, Any], approved: bool = False) -> Dict[str, Any]:
        tool = self.tools[tool_name]
        if tool.sensitive and not approved:
            return {'status': 'denied', 'reason': 'human_approval_required'}
        return {'status': 'ok', 'result': tool.func(args)}

def weather_tool(args):
    city = args.get('city', 'unknown')
    return {'city': city, 'temp_c': 27, 'condition': 'clear'}

def close_ticket_tool(args):
    return {'ticket_id': args.get('ticket_id'), 'closed': True}

server = MCPServer(
    name='support-mcp',
    tools={
        'get_weather': MCPTool('get_weather', 'Fetch weather by city', weather_tool, sensitive=False),
        'close_ticket': MCPTool('close_ticket', 'Close support ticket', close_ticket_tool, sensitive=True),
    },
)
server.list_tools()

[{'name': 'get_weather',
  'description': 'Fetch weather by city',
  'sensitive': False},
 {'name': 'close_ticket',
  'description': 'Close support ticket',
  'sensitive': True}]

In [3]:
server.call_tool('get_weather', {'city': 'Bengaluru'})

{'status': 'ok',
 'result': {'city': 'Bengaluru', 'temp_c': 27, 'condition': 'clear'}}

In [4]:
server.call_tool('close_ticket', {'ticket_id': 'T-1042'}, approved=False)

{'status': 'denied', 'reason': 'human_approval_required'}

In [5]:
server.call_tool('close_ticket', {'ticket_id': 'T-1042'}, approved=True)

{'status': 'ok', 'result': {'ticket_id': 'T-1042', 'closed': True}}

## 2) Minimal A2A Simulation

In [6]:
class TaskState(str, Enum):
    PENDING = 'pending'
    RUNNING = 'running'
    DONE = 'done'

@dataclass
class A2ATask:
    task_id: str
    requester: str
    assignee: str
    payload: Dict[str, Any]
    state: TaskState = TaskState.PENDING
    updates: List[str] = field(default_factory=list)

@dataclass
class Agent:
    name: str
    capabilities: List[str]

    def discover(self) -> Dict[str, Any]:
        return {'agent': self.name, 'capabilities': self.capabilities}

    def start_task(self, requester: str, payload: Dict[str, Any]) -> A2ATask:
        t = A2ATask(task_id=str(uuid.uuid4())[:8], requester=requester, assignee=self.name, payload=payload)
        t.state = TaskState.RUNNING
        t.updates.append('accepted')
        return t

    def progress(self, task: A2ATask) -> A2ATask:
        task.updates.append('processing')
        time.sleep(0.05)
        task.state = TaskState.DONE
        task.updates.append('completed')
        return task

finance_agent = Agent('finance-agent', ['invoice_lookup', 'refund_policy', 'escalate_billing'])
orchestrator = Agent('orchestrator-agent', ['route_request', 'aggregate_response'])
finance_agent.discover()

{'agent': 'finance-agent',
 'capabilities': ['invoice_lookup', 'refund_policy', 'escalate_billing']}

In [7]:
task = finance_agent.start_task(requester=orchestrator.name, payload={'ticket': 'BILL-77', 'action': 'refund_policy'})
finance_agent.progress(task)
{'task_id': task.task_id, 'state': task.state, 'updates': task.updates}

{'task_id': 'f7ea67f6',
 'state': <TaskState.DONE: 'done'>,
 'updates': ['accepted', 'processing', 'completed']}

## 3) Combined MCP + A2A Pattern

In [8]:
combined_flow = {
    'step_1': 'orchestrator uses MCP tools for local context',
    'step_2': 'orchestrator delegates specialist work via A2A',
    'step_3': 'orchestrator validates returned payload against policy',
    'step_4': 'human approval for sensitive actions',
}
combined_flow

{'step_1': 'orchestrator uses MCP tools for local context',
 'step_2': 'orchestrator delegates specialist work via A2A',
 'step_3': 'orchestrator validates returned payload against policy',
 'step_4': 'human approval for sensitive actions'}

## Case Studies & Exceptions
1. Customer support mesh: MCP for local systems, A2A for specialist routing.
2. Supply chain coordination: A2A async tasks plus MCP local ERP actions.
3. Exception: single-agent systems often only need MCP.

## Interview Questions & Answers
1. **Q:** MCP vs A2A? **A:** MCP integrates tools/context; A2A enables cross-agent collaboration.
2. **Q:** Why async in A2A? **A:** Long-running tasks need explicit lifecycle state.
3. **Q:** What is capability discovery? **A:** Protocol-level listing of supported operations.
4. **Q:** Why approval gates? **A:** Prevent unsafe autonomous actions.
5. **Q:** Can MCP replace A2A? **A:** No.
6. **Q:** Can A2A replace MCP? **A:** No.
7. **Q:** Key security control? **A:** Scoped authorization per operation.
8. **Q:** How prevent loops? **A:** Step budgets and terminal-state checks.
9. **Q:** Why validate remote outputs? **A:** Treat as untrusted until checked.
10. **Q:** One-line advice? **A:** Protocol boundaries are security boundaries.